# Session 01: The Modern Data Stack & Fast Interactive Telemetry
*Python for Data Analysis — Applied Engineering Edition*

> **Navigation:** [<< Course Roadmap](./00_roadmap_and_orientation.ipynb) | **Current: Session 01** | [Next: Session 02 >>](./session_02_numpy_arrays.ipynb)

---

## Welcome to Real-World Data Analysis

When you work with modern IoT systems and building automation, data does not arrive as tidy, isolated numbers. It flows continuously from hundreds of sensors: chiller supply temperatures, electrical sub-meters, flow meters, and environmental monitors. 

To work with this data effectively, we need to understand the practical programming tools and engineering patterns of Python's scientific ecosystem.

Let's start by seeing our modern toolchain in action with an end-to-end telemetry pipeline.

### Seeing the Whole Picture: A First Telemetry Pipeline
Run the cell below (`Shift + Enter`). This script checks our core libraries (`pandas`, `numpy`) and immediately processes an initial snapshot of building sensor data:

In [ ]:
import sys
import numpy as np
import pandas as pd

print(f"Verified Environment: Python {sys.version.split()[0]} | NumPy {np.__version__} | pandas {pd.__version__}")

# Configure clean console display formatting
pd.options.display.max_columns = 10
pd.options.display.max_rows = 15

# A realistic building telemetry snapshot
telemetry = pd.DataFrame({
    'timestamp': pd.to_datetime(['2026-03-01 08:00', '2026-03-01 08:00', '2026-03-01 08:15', '2026-03-01 08:15']),
    'sensor_id': ['chiller_01', 'meter_main', 'chiller_01', 'meter_main'],
    'reading': [42.5, 310.2, 44.1, 315.8],
    'unit': ['deg_c', 'kw', 'deg_c', 'kw']
})

print("\nRaw Telemetry Records:")
display(telemetry)

# Slicing out peak demand for our main power meter
power_only = telemetry[telemetry['sensor_id'] == 'meter_main']
print(f"\nPeak Electrical Demand: {power_only['reading'].max()} kW")


---

## 1. What Exactly is "Structured Data"?

Notice the table we just printed above. That is our primary subject: **Structured Data**.

When engineers say "data," they often lump together everything from raw text logs to image pixels. In analytics, we focus on datasets that can be mapped into concrete, organized forms:

1. **Tabular Data:** Records organized into rows and columns, where each column holds a specific data type (timestamps, device IDs, numeric floats). Think CSV files, Parquet columnar logs, and SQL relational tables.
2. **Multidimensional Arrays:** Dense numerical grids where every value shares the exact same numeric type. This is what NumPy's `ndarray` provides—perfect for multi-channel sensor signals sampled over time.
3. **Relational Tables:** Separate tables tied together by key columns (for instance, joining a table of hourly kWh readings to a table of building zone square-footage using a `room_id` key).
4. **Time Series:** Observations explicitly ordered and indexed by timestamps, whether regularly spaced (every 15 minutes) or event-driven (anomalous spike alerts).

Even when you collect messy text logs or telemetry JSON payloads, your first job as an analyst is almost always to parse and shape them into one of these structured representations.

---

## 2. Why Did Python Win in Data Science?

Python is an interpreted scripting language created in 1991. Why did it become the dominant language for high-performance data engineering and machine learning?

### The "Two-Language Problem"
Historically, companies suffered from an expensive organizational divide:
* Researchers prototyped models in specialized mathematical languages like R, SAS, or MATLAB.
* When an algorithm was ready for production, software engineers had to spend months rewriting the entire codebase from scratch in C++, Java, or C#.

Python solved this problem by being **fast to write** for exploration, yet **robust enough** to run production web servers and data pipelines. The research team and the software engineering team can finally speak the same language.

### Python as the "Glue"
The secret to Python's speed in data science is that Python itself doesn't do the heavy numeric calculations in pure Python bytecode! 

For decades, the scientific community perfected high-performance linear algebra and matrix libraries in C and Fortran. Python acts as the **high-level glue code**:
* The heavy lifting happens down in low-level compiled C memory buffers at raw processor speeds.
* The clean, expressive orchestration happens in Python.

### The Memory Reality: Python Lists vs. NumPy Buffers
A standard Python list stores pointers to scattered Python objects in memory, with each number carrying 24 to 28 bytes of object overhead. A NumPy array, by contrast, stores raw numbers packed contiguously in C memory—taking just 8 bytes per float.

Run the cell below to compare memory footprint and calculation speed on your own machine:

In [ ]:
import sys
import timeit
import numpy as np

n = 200_000

# 1. Memory footprint comparison
py_list = [float(x) for x in range(n)]
np_arr = np.arange(n, dtype=np.float64)

np_bytes = np_arr.nbytes
py_ptr_bytes = sys.getsizeof(py_list)
py_total_est = py_ptr_bytes + (24 * n)  # Each float object adds ~24 bytes of header memory

print(f"NumPy contiguous C-buffer:    {np_bytes / (1024**2):.2f} MB")
print(f"Python list (estimated total): {py_total_est / (1024**2):.2f} MB")

# 2. Speed comparison: Converting Celsius to Fahrenheit
t_py = timeit.timeit('[c * 1.8 + 32 for c in py_list]', globals=locals(), number=30)
t_np = timeit.timeit('np_arr * 1.8 + 32', globals=locals(), number=30)

print(f"\nPython List Comprehension Time: {t_py:.4f}s")
print(f"NumPy Vectorized C-Loop Time:   {t_np:.4f}s")
print(f"Actual Measured Speedup:        {t_py / t_np:.1f}x faster!")


---

## 3. Understanding Vectorization: How Math Applies to Whole Arrays

In the benchmark cell above, notice this specific expression:
```python
np_arr * 1.8 + 32
```
This is **Vectorization**. Instead of writing a `for` loop to inspect and multiply each number one by one, NumPy applies arithmetic operators directly to the entire array simultaneously.

Let's see how this works step by step:

### A Quick Mental Check
Examine the small array below. What will NumPy output when we subtract `0.5` from every element and then multiply by `100`?
* Formulate your mental prediction before running the cell!

In [ ]:
raw_voltages = np.array([0.5, 0.75, 1.0])

# Mental Prediction: (0.5 - 0.5)*100 = 0.0, (0.75 - 0.5)*100 = 25.0, (1.0 - 0.5)*100 = 50.0
converted_temps = (raw_voltages - 0.5) * 100
print("Resulting Temperatures (°C):", converted_temps)


### Hands-On: Adapting the Conversion
In the code cell below, modify the formula to convert these Celsius temperatures to Fahrenheit:
$$\text{temp\_f} = \text{converted\_temps} \times 1.8 + 32$$
* Change the operation and run the cell:

In [ ]:
# Modify the line below to compute temp_f:
temp_f = converted_temps * 1.8 + 32
print("Resulting Temperatures (°F):", temp_f)


### Hands-On: Completing the Calibration Formula
Many temperature sensors (like the TMP36 analog sensor) output voltage between `0.5V` (for 0°C) and `1.5V` (for 100°C).
The calibration formula is:
$$\text{temp\_c} = (\text{voltage} - 0.5) \times 100$$

NumPy also gives you built-in array aggregation methods like `.min()` and `.max()` to inspect the range of an array.

Complete the blanks below (replace `___` with the correct operator/number):

In [ ]:
test_voltages = np.array([0.65, 0.72, 0.81, 0.95])

# Complete the formula based on the example above:
# temp_c = (test_voltages - 0.5) * 100
temp_c = (test_voltages - 0.5) * 100

print("Minimum Temperature:", temp_c.min())
print("Maximum Temperature:", temp_c.max())


### Guided Exercise: Calibrating 100,000 Sensor Readings
Now apply this to a realistic stream of 100,000 voltage samples from building environmental sensors.

**Your Task:**
1. In the cell below, calculate `sensor_temps = (stream_voltages - 0.5) * 100`.
2. Compute `min_temp = sensor_temps.min()` and `max_temp = sensor_temps.max()`.
3. Run the cell to execute the automated verification test (`assert`)!

In [ ]:
np.random.seed(42)
# Realistic building sensor voltage range: 0.65V (15°C) to 0.85V (35°C)
stream_voltages = np.random.uniform(0.65, 0.85, size=100_000)

# [YOUR CODE HERE]
sensor_temps = (stream_voltages - 0.5) * 100
min_temp = sensor_temps.min()
max_temp = sensor_temps.max()

# Automated Verification Check:
assert round(min_temp, 1) == 15.0, f"Expected min ~15.0, got {min_temp:.1f}"
assert round(max_temp, 1) == 35.0, f"Expected max ~35.0, got {max_temp:.1f}"
print("[PASS] Excellent! Verification passed. All 100,000 readings calibrated in microseconds.")
print(f"Temperature Range: {min_temp:.2f}°C to {max_temp:.2f}°C")


---

## 4. The Interactive Superpower: The Execute-Explore Workflow

When you build traditional web applications, you follow an **Edit-Compile-Run** cycle: you edit source files, restart your server, and test. 

Data analysis is fundamentally different. If your telemetry dataset takes 5 minutes to parse from disk, you cannot afford to restart your program every time you want to try a new chart or calculate a new summary statistic.

This brings us to IPython and Jupyter's core philosophy: the **Execute-Explore Workflow**.
* You load your telemetry into memory once.
* The **IPython Kernel** keeps that data resident in RAM.
* You can interactively test transformations, run quick calculations, and inspect variables on the fly.

### Inspecting Code Without Leaving Your Notebook
A huge productivity booster in IPython is **Introspection**:
* Append a single question mark `?` to any function to immediately read its docstring and argument signature.
* Append `??` to see the actual Python source code of that function.
* Use wildcards (like `pd.*read*?`) to search for functions when you cannot remember the exact name.

Run the cell below to see how easy it is to inspect our own helper functions:

In [ ]:
import pandas as pd

def compute_sensor_drift(raw_readings, baseline=20.0):
    """
    Calculate how far temperature readings have drifted from a desired target.
    
    Parameters:
        raw_readings (array-like): Measured telemetry values.
        baseline (float): Target reference temperature in Celsius (default 20.0).
        
    Returns:
        array-like: Absolute deviation from baseline.
    """
    return np.abs(np.asarray(raw_readings) - baseline)

# Type the function name followed by '?' to inspect its documentation inline:
compute_sensor_drift?


### Try It Yourself
Use the code cell below to practice inline introspection:
1. Run `pd.*read*?` to see all data reading functions available in pandas.
2. Run `pd.read_parquet?` to discover what engine pandas uses by default to read Parquet files.

In [ ]:
# [YOUR CODE HERE]
# 1. Search for reading functions:
# pd.*read*?

# 2. Inspect read_parquet:
# pd.read_parquet?


---

## 5. Learning Through Deliberate Mistakes

Before we move to our main exercise, let's look at two mistakes almost every data analyst makes early on. Learning to recognize real tracebacks immediately will save you hours of debugging down the road.

### Trap 1: The Magic Command Placement Rule
IPython provides special commands prefixed with `%` (Line Magics) or `%%` (Cell Magics).
* A line magic (like `%timeit`) applies to a single line.
* A cell magic (like `%%time`) measures the entire cell.

**The Rule:** A cell magic `%%` **must be placed on the absolute first line** (Line 1, Column 1). If you place a comment or blank line above it, the IPython parser will fail with a `UsageError`.

In [ ]:
# Notice that %%time is right on Line 1 with no comments above it:
%%time
import time
time.sleep(0.02)
print("Cell execution finished!")

# If we had put '# Benchmark comment' on Line 1 and %%time on Line 2, 
# IPython would have thrown: UsageError: Line magic function `%%time` not found.


### Trap 2: The Danger of Wildcard Imports
You will sometimes see beginner tutorials write `from numpy import *` or `from pandas import *`. 

**Never do this.** When you import with a wildcard, NumPy's functions silently overwrite Python's built-in functions. 
For example, Python's built-in `sum` accepts a `start=` offset parameter, while NumPy's `sum` does not. Run the cell below to see what happens:

In [ ]:
# Python's built-in sum works like this:
values = [10, 20, 30]
print("Built-in sum with start offset:", sum(values, start=5))

# But if you did 'from numpy import *', calling sum(values, start=5) 
# crashes with: TypeError: sum() got an unexpected keyword argument 'start'
# That's why we always use the universal community convention: import numpy as np


---

## 6. Real-World Challenge: 24-Hour Building Energy Audit

Now let's bring everything together in a realistic engineering scenario without hints.

Imagine you are handed 24 hours of sub-meter telemetry measuring power consumption (in Kilowatts) across three distinct building zones: `Zone_East`, `Zone_West`, and `Zone_Core`.

Run the cell below to load our test dataset:

In [ ]:
import numpy as np
import pandas as pd

# Generate realistic 24-hour telemetry
np.random.seed(42)
hours = pd.date_range("2026-03-01", periods=24, freq="1h")
zones = ["Zone_East", "Zone_West", "Zone_Core"]

records = []
for h in hours:
    for z in zones:
        # Baseline load with random fluctuations between 10 kW and 45 kW
        p = np.random.uniform(10.0, 45.0)
        records.append({'timestamp': h, 'zone': z, 'power_kw': round(p, 2)})

sensor_df = pd.DataFrame(records)
print(f"Loaded {len(sensor_df)} telemetry records.")
display(sensor_df.head())


### Your Engineering Challenge
In the code cell below, complete the following analysis:
1. **Check the schema:** Run `sensor_df.info()` directly (do NOT wrap it in `print()`) to verify the column types and memory footprint.
2. **Filter high-demand periods:** Filter `sensor_df` to isolate only records from `"Zone_Core"` where `power_kw > 35.0`.
3. **Compute total consumption:** Group the data by zone and compute the total power used across the full 24 hours: `sensor_df.groupby('zone')['power_kw'].sum()`.
4. **Identify the highest consumer:** Output which zone consumed the greatest total power.

In [ ]:
# [YOUR CODE HERE]
# Task 1: Check info

# Task 2: Filter Zone_Core with power_kw > 35.0

# Task 3: Calculate total consumption by zone

# Task 4: Output the highest consuming zone


---

## 7. Key Takeaways & Review Routine

Congratulations on finishing Session 01! Here are the core concepts that should stick with you:

1. **Structured data** (tables, matrices, time series) is the universal currency of data analysis.
2. Python acts as **glue**—it provides an elegant, expressive interface on top of fast, compiled C routines.
3. **Vectorization** eliminates `for` loops by performing arithmetic across entire contiguous C-memory arrays simultaneously.
4. The **Execute-Explore workflow** keeps data resident in memory, letting you iterate in seconds rather than restarting scripts from scratch.

### Quick Retrieval Check (Task ➔ Code)
Before closing this notebook, test yourself on these three active recall prompts:
* **Prompt 1:** How do you print a DataFrame's columns, types, and non-null counts cleanly without a trailing `None`?
  * *Answer:* `df.info()`
* **Prompt 2:** How do you inspect a function's full Python source code in IPython?
  * *Answer:* `func_name??`
* **Prompt 3:** In modern pandas (3.0+), what method replaces `applymap` for element-wise transformations across all DataFrame cells?
  * *Answer:* `df.map(lambda x: ...)`

### Coming Up in Session 02:
In our next session, we will dive deep into **NumPy Multi-Dimensional Arrays (`ndarray`)**: understanding memory strides, slicing multi-channel sensor matrices, and mastering **Broadcasting** to calibrate thousands of sensors simultaneously without a single loop!

👉 **Next Step:** Open **[Session 02: Multi-Channel Signals & NumPy ndarrays](./session_02_numpy_arrays.ipynb)** to continue your journey.